<a href="https://colab.research.google.com/github/buse-toklu/CENG467_Midterm/blob/main/2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — Install Dependencies
!pip install datasets seqeval sklearn-crfsuite transformers accelerate torchcrf -q

In [ ]:
import json, random, time, warnings
from collections import Counter
from typing import Dict, List, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

from datasets import load_dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    get_linear_schedule_with_warmup,
)

import sklearn_crfsuite
!pip install --upgrade --force-reinstall torchcrf -q
from torchcrf import CRF
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# Cell 3 — Label Map
label_list = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}
NUM_LABELS = len(label_list)

print(id2label)

In [ ]:
# Cell 4 — Load Dataset
dataset_ner = load_dataset("lhoestq/conll2003")

def convert_split(split_data):
    records = []
    for row in split_data:
        tokens = row["tokens"]
        tags   = [label_list[t] for t in row["ner_tags"]]
        records.append({"tokens": tokens, "tags": tags})
    return records

train = convert_split(dataset_ner["train"])
val   = convert_split(dataset_ner["validation"])
test  = convert_split(dataset_ner["test"])

print(f"Train : {len(train):,} sentences")
print(f"Val   : {len(val):,} sentences")
print(f"Test  : {len(test):,} sentences")

In [ ]:
# Cell 5 — Describe Dataset & Validate BIO Alignment

# Annotation structure
print("CoNLL-2003 Annotation Structure")
print("  Format  : one token per line, BIO (IOB2) scheme")
print("  Columns : word | POS tag | chunk tag | NER tag")
print("  Entities: PER, ORG, LOC, MISC")
print("  B- = span start,  I- = span continuation,  O = outside")

# Label distribution
tag_counts = Counter(tag for r in train for tag in r["tags"])
print("\nLabel distribution (train):")
for tag, cnt in sorted(tag_counts.items()):
    print(f"  {tag:<8} {cnt:>7,}")

# Sample sentence
sample = train[0]
print("\nExample sentence:")
print(f"  {'TOKEN':<16} TAG")
print(f"  {'-'*25}")
for tok, tag in zip(sample["tokens"], sample["tags"]):
    print(f"  {tok:<16} {tag}")

# BIO alignment check
errors = 0
for r in train:
    prev = "O"
    for tag in r["tags"]:
        if tag.startswith("I-"):
            etype = tag[2:]
            if not (prev == f"B-{etype}" or prev == f"I-{etype}"):
                errors += 1
        prev = tag
print(f"\nBIO alignment: {errors} violations ({'CLEAN' if errors == 0 else 'WARNING'})")

In [ ]:
# Cell 6 — CRF Feature Extraction

def word_features(sentence, i):
    word = sentence[i]
    feats = {
        "bias":           1.0,
        "word.lower()":   word.lower(),
        "word[-3:]": word[-3:], "word[-2:]": word[-2:], "word[:3]": word[:3],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
        "word.has_hyphen": "-" in word,
        "BOS": i == 0,
        "EOS": i == len(sentence) - 1,
    }
    if i > 0:
        w = sentence[i-1]
        feats.update({"-1:lower": w.lower(), "-1:title": w.istitle(), "-1:upper": w.isupper()})
    if i < len(sentence) - 1:
        w = sentence[i+1]
        feats.update({"+1:lower": w.lower(), "+1:title": w.istitle(), "+1:upper": w.isupper()})
    if i > 1:               feats["-2:lower"] = sentence[i-2].lower()
    if i < len(sentence)-2: feats["+2:lower"] = sentence[i+2].lower()
    return feats

def records_to_features(records):
    return [[word_features(r["tokens"], i) for i in range(len(r["tokens"]))] for r in records]

def records_to_labels(records):
    return [r["tags"] for r in records]

print("CRF feature functions defined.")

In [ ]:
# Cell 7 — Train & Evaluate CRF

def evaluate(y_true, y_pred, model_name="Model"):
    p  = precision_score(y_true, y_pred)
    r  = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"\n[{model_name}]")
    print(f"  Precision : {p:.4f}")
    print(f"  Recall    : {r:.4f}")
    print(f"  F1        : {f1:.4f}")
    print(classification_report(y_true, y_pred, digits=4))
    return {"model": model_name, "precision": round(p,4), "recall": round(r,4), "f1": round(f1,4)}


X_train = records_to_features(train)
y_train = records_to_labels(train)
X_val   = records_to_features(val)
y_val   = records_to_labels(val)

crf_model = sklearn_crfsuite.CRF(
    algorithm="lbfgs", c1=0.1, c2=0.1,
    max_iterations=100, all_possible_transitions=True
)
t0 = time.time()
crf_model.fit(X_train, y_train)
print(f"CRF trained in {time.time()-t0:.1f}s")

y_pred_crf = crf_model.predict(X_val)
crf_met = evaluate(y_val, y_pred_crf, model_name="CRF")

In [ ]:
# Cell 8 — CRF Error Analysis

def get_spans(seq):
    spans, start, etype = [], None, None
    for i, tag in enumerate(seq):
        if tag.startswith("B-"):
            if start is not None: spans.append((etype, start, i-1))
            start, etype = i, tag[2:]
        elif tag.startswith("I-"):
            if start is None: start, etype = i, tag[2:]
        else:
            if start is not None:
                spans.append((etype, start, i-1)); start, etype = None, None
    if start is not None: spans.append((etype, start, len(seq)-1))
    return spans

def error_analysis(y_true, y_pred, model_name):
    print(f"\n[ERROR ANALYSIS] {model_name}")
    print("-" * 45)
    boundary_errors = 0
    type_confusion  = Counter()
    false_positives = Counter()
    false_negatives = Counter()

    for sent_true, sent_pred in zip(y_true, y_pred):
        true_spans = set(get_spans(sent_true))
        pred_spans = set(get_spans(sent_pred))
        matched_t, matched_p = set(), set()
        for t_ent, t_s, t_e in true_spans:
            for p_ent, p_s, p_e in pred_spans:
                if t_s == p_s and t_e == p_e:
                    if t_ent != p_ent: type_confusion[(t_ent, p_ent)] += 1
                    matched_t.add((t_ent, t_s, t_e)); matched_p.add((p_ent, p_s, p_e))
                elif t_s <= p_e and p_s <= t_e:
                    boundary_errors += 1
                    matched_t.add((t_ent, t_s, t_e)); matched_p.add((p_ent, p_s, p_e))
        for span in true_spans - matched_t: false_negatives[span[0]] += 1
        for span in pred_spans - matched_p: false_positives[span[0]] += 1

    print(f"  Boundary errors : {boundary_errors}")
    print(f"  Type confusion  :")
    for (t, p), cnt in type_confusion.most_common(5): print(f"    {t} -> {p}: {cnt}")
    print(f"  False positives :")
    for etype, cnt in false_positives.most_common(): print(f"    {etype}: {cnt}")
    print(f"  False negatives :")
    for etype, cnt in false_negatives.most_common(): print(f"    {etype}: {cnt}")
    return {
        "boundary_errors": boundary_errors,
        "type_confusion":  {f"{t}->{p}": c for (t,p),c in type_confusion.items()},
        "false_positives": dict(false_positives),
        "false_negatives": dict(false_negatives),
    }


all_errors = {}
all_errors["CRF"] = error_analysis(y_val, y_pred_crf, "CRF")

In [ ]:
# Cell 9 — Vocabulary & Dataset Class

def build_vocab(records, min_freq=1):
    counter = Counter(t for r in records for t in r["tokens"])
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab


class NERDataset(Dataset):
    def __init__(self, records, vocab, label2id, max_len=128):
        self.samples = []
        for r in records:
            ids  = [vocab.get(t, vocab["<UNK>"]) for t in r["tokens"]]
            lids = [label2id[t] for t in r["tags"]]
            mask = [1] * len(ids)
            ids  = ids[:max_len];  lids = lids[:max_len];  mask = mask[:max_len]
            pad  = max_len - len(ids)
            ids += [0]*pad; lids += [label2id["O"]]*pad; mask += [0]*pad
            self.samples.append((
                torch.tensor(ids,  dtype=torch.long),
                torch.tensor(lids, dtype=torch.long),
                torch.tensor(mask, dtype=torch.uint8),
            ))
    def __len__(self):        return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


print("NERDataset defined.")

In [ ]:
# Cell 10 — BiLSTM-CRF Model

class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, num_labels,
                 embed_dim=100, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim//2,
                            num_layers=num_layers, batch_first=True,
                            bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, num_labels)
        self.crf     = CRF(num_labels, batch_first=True)

    def forward(self, x, tags=None, mask=None):
        emb       = self.dropout(self.embedding(x))
        out, _    = self.lstm(emb)
        emissions = self.fc(self.dropout(out))
        if tags is not None:
            return -self.crf(emissions, tags, mask=mask, reduction="mean")
        return self.crf.decode(emissions, mask=mask)


print("BiLSTM_CRF defined.")

In [ ]:
# Cell 11 — Train BiLSTM-CRF
EPOCHS_BILSTM = 5
BATCH_SIZE    = 32

vocab    = build_vocab(train)
dl_train = DataLoader(NERDataset(train, vocab, label2id), batch_size=BATCH_SIZE, shuffle=True)
dl_val   = DataLoader(NERDataset(val,   vocab, label2id), batch_size=BATCH_SIZE)

bilstm_model = BiLSTM_CRF(len(vocab), NUM_LABELS).to(DEVICE)
opt   = Adam(bilstm_model.parameters(), lr=1e-3)
sched = get_linear_schedule_with_warmup(
    opt,
    num_warmup_steps=len(dl_train),
    num_training_steps=len(dl_train) * EPOCHS_BILSTM
)

best_f1, best_state = 0.0, None

for epoch in range(1, EPOCHS_BILSTM + 1):
    bilstm_model.train()
    total_loss = 0.0
    for x, y, mask in dl_train:
        x, y, mask = x.to(DEVICE), y.to(DEVICE), mask.to(DEVICE)
        loss = bilstm_model(x, tags=y, mask=mask.bool())
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(bilstm_model.parameters(), 1.0)
        opt.step(); sched.step()
        total_loss += loss.item()

    bilstm_model.eval()
    y_true_all, y_pred_all = [], []
    with torch.no_grad():
        for x, y, mask in dl_val:
            x, mask = x.to(DEVICE), mask.to(DEVICE)
            preds = bilstm_model(x, mask=mask.bool())
            for pred_seq, true_seq, m in zip(preds, y.tolist(), mask.tolist()):
                length = sum(m)
                y_true_all.append([id2label[t] for t in true_seq[:length]])
                y_pred_all.append([id2label[p] for p in pred_seq[:length]])

    f1 = f1_score(y_true_all, y_pred_all)
    print(f"Epoch {epoch}/{EPOCHS_BILSTM}  loss={total_loss/len(dl_train):.4f}  val_f1={f1:.4f}")
    if f1 > best_f1:
        best_f1    = f1
        best_state = {k: v.cpu().clone() for k, v in bilstm_model.state_dict().items()}

bilstm_model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

# Final evaluation on val
bilstm_model.eval()
y_true_bl, y_pred_bl = [], []
with torch.no_grad():
    for x, y, mask in dl_val:
        x, mask = x.to(DEVICE), mask.to(DEVICE)
        preds = bilstm_model(x, mask=mask.bool())
        for pred_seq, true_seq, m in zip(preds, y.tolist(), mask.tolist()):
            length = sum(m)
            y_true_bl.append([id2label[t] for t in true_seq[:length]])
            y_pred_bl.append([id2label[p] for p in pred_seq[:length]])

bilstm_met = evaluate(y_true_bl, y_pred_bl, model_name="BiLSTM-CRF")

In [ ]:
# Cell 12 — BiLSTM-CRF Error Analysis
all_errors["BiLSTM-CRF"] = error_analysis(y_true_bl, y_pred_bl, "BiLSTM-CRF")

In [ ]:
# Cell 13 — Tokenisation & Label Alignment
MODEL_CHECKPOINT = "bert-base-cased"

def tokenize_and_align(examples, tokenizer):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True, is_split_into_words=True,
        max_length=128, padding="max_length"
    )
    labels = []
    for i, tag_ids in enumerate(examples["ner_tags"]):
        word_ids     = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids    = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(tag_ids[word_id])
            else:
                label_ids.append(-100)
            prev_word_id = word_id
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_preds  = [[label_list[pr] for pr, lb in zip(pred, label) if lb != -100]
                   for pred, label in zip(preds, labels)]
    true_labels = [[label_list[lb] for lb in label if lb != -100]
                   for label in labels]
    return {
        "precision": precision_score(true_labels, true_preds),
        "recall":    recall_score(true_labels, true_preds),
        "f1":        f1_score(true_labels, true_preds),
    }

print("BERT helpers defined.")

In [ ]:
# Cell 14 — Fine-tune BERT
EPOCHS_BERT = 3

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
tokenized = dataset_ner.map(lambda b: tokenize_and_align(b, tokenizer), batched=True)

bert_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

args = TrainingArguments(
    output_dir="./bert_ner_ckpt",
    num_train_epochs=EPOCHS_BERT,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    seed=SEED,
    report_to="none"
)

trainer = Trainer(
    model=bert_model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

preds_out  = trainer.predict(tokenized["validation"])
raw_preds  = np.argmax(preds_out.predictions, axis=2)
raw_labels = preds_out.label_ids

y_true_bert, y_pred_bert = [], []
for pred_seq, label_seq in zip(raw_preds, raw_labels):
    y_true_bert.append([label_list[l] for l in label_seq if l != -100])
    y_pred_bert.append([label_list[p] for p, l in zip(pred_seq, label_seq) if l != -100])

bert_met = evaluate(y_true_bert, y_pred_bert, model_name="BERT")

In [ ]:
# Cell 15 — BERT Error Analysis
all_errors["BERT"] = error_analysis(y_true_bert, y_pred_bert, "BERT")

In [ ]:
# Cell 16 — Comparison Table
all_metrics = [crf_met, bilstm_met, bert_met]

print("=" * 58)
print("  FINAL COMPARISON")
print("=" * 58)
print(f"  {'Model':<20} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("  " + "-" * 50)
for m in all_metrics:
    print(f"  {m['model']:<20} {m['precision']:>10.4f} {m['recall']:>10.4f} {m['f1']:>10.4f}")

In [ ]:
# Cell 17 — Save Results
results = {"metrics": all_metrics, "error_analysis": all_errors}
with open("ner_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved to ner_results.json")
print(json.dumps(all_metrics, indent=2))